## Multi-Agent System (MAS) — DIY

**키키테크 사내 Q&A 챗봇**을 멀티 에이전트로 구축합니다.

이번 실습은 코드를 바로 채우는 것으로 시작하지 않습니다.
먼저 **어떤 구조로 만들지 직접 설계**하고, 그 설계를 코드로 옮겨봅니다.

키키테크에는 다음 데이터가 있습니다.

| 파일 | 형식 | 담고 있는 정보 |
|------|------|----------------|
| 키키테크_임직원및프로젝트현황.xlsx | 엑셀 | 직원 이름, 부서, 연락처, 담당업무, 프로젝트 |
| 키키테크_AI솔루션_제품카탈로그.pdf | PDF | 제품 종류, 기능, 요금제 |
| 키키테크_회사소개.pptx | PPT | 회사 개요, 비전, 조직도 |
| 키키테크_사내규정_행동강령.docx | Word | 근무 규정, 복리후생, 행동강령 |

챗봇은 이런 질문들에 답할 수 있어야 합니다.

```
"이서연 책임의 내선번호가 뭐야?"
"TalkBridge Enterprise 요금제가 어떻게 돼?"
"재택근무는 일주일에 몇 번까지 가능해?"
```

## Step 1. 설계해보기

구현 전에 먼저 구조를 생각해 봅시다.

---

**Q1.** 위 질문 세 개를 에이전트 1개가 모두 처리하는 것보다 여러 개로 나누는 게 좋은 이유가 뭘까요?

---

**Q2.** 몇 개의 에이전트로 나누겠어요? 각 에이전트의 역할과 사용할 데이터를 적어보세요.

| 에이전트 이름 | 담당하는 질문 유형 | 필요한 데이터 |
|---|---|---|
| | | |
| | | |
| | | |

---

**Q3.** Supervisor는 어떤 기준으로 알맞은 에이전트를 고를까요?

---

> 설계를 마쳤으면 아래 셀에 작성하고, Step 2부터 구현해 봅시다.
> 구현이 끝나면 내 설계와 실제 코드를 비교해 보세요.

✏️ **설계 작성 공간**

**Q1 답:**



**Q2 답:**

| 에이전트 이름 | 담당하는 질문 유형 | 필요한 데이터 |
|---|---|---|
| | | |
| | | |
| | | |

**Q3 답:**



## Step 2. 환경 설정

In [ ]:
import os
import uuid
import pandas as pd
from dotenv import load_dotenv, find_dotenv

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool

from rag_retriever import RAGRetriever

load_dotenv(find_dotenv())

credential_key = os.getenv('credential_key')
send_system_name = os.getenv('send_system_name')
model_name = os.getenv('model')
api_base_url = os.getenv('api_base_url')
user_id = os.getenv('user_id')

os.environ['OPENAI_API_KEY'] = 'api_key'

llm = ChatOpenAI(
    model=model_name,
    base_url=api_base_url,
    default_headers={
        'x-dep-ticket': credential_key,
        'Send-System-Name': send_system_name,
        'User-Id': user_id,
        'User-Type': 'AD_ID',
        'Prompt-Msg-Id': str(uuid.uuid4()),
        'Completion-Msg-Id': str(uuid.uuid4())
    },
    temperature=0.7,
)

## Step 3. 검색 기능 준비 (제공됨)

각 도구 내부에서 사용할 검색 기능입니다. 직접 수정할 필요 없이 그대로 실행하세요.

- **임직원 검색**: 엑셀 파일을 pandas로 직접 읽어 이름으로 조회합니다.
- **제품·규정 검색**: `RAGRetriever`로 문서를 임베딩한 뒤 유사도 검색을 합니다.

In [ ]:
# 임직원 검색 — pandas로 엑셀 직접 조회
def _search_employee_by_name(name: str) -> str:
    df = pd.read_excel('../04_RAG/data/키키테크_임직원및프로젝트현황.xlsx', header=1)
    result = df[df['성명'] == name]
    if result.empty:
        return f"'{name}' 이름의 임직원을 찾을 수 없습니다."
    row = result.iloc[0]
    return '\n'.join(f'{col}: {row[col]}' for col in df.columns)


# 제품 검색 — PDF + PPT RAG
product_rag = RAGRetriever()
product_rag.add_file('../04_RAG/data/키키테크_AI솔루션_제품카탈로그.pdf')
product_rag.add_file('../04_RAG/data/키키테크_회사소개.pptx')
product_rag.build_retriever()

def _search_product_docs(query: str) -> str:
    results = product_rag.search(query)
    return '\n\n---\n\n'.join(doc.page_content for doc in results)


# 규정 검색 — Word RAG
policy_rag = RAGRetriever()
policy_rag.add_file('../04_RAG/data/키키테크_사내규정_행동강령.docx')
policy_rag.build_retriever()

def _search_policy_docs(query: str) -> str:
    results = policy_rag.search(query)
    return '\n\n---\n\n'.join(doc.page_content for doc in results)


print('검색 기능 준비 완료')

## Step 4. 도구(Tool) 만들기

구현부는 제공되어 있습니다. **docstring만 직접 작성하세요.**

docstring은 단순한 설명이 아닙니다 — LLM이 이 텍스트를 읽고 *언제, 어떻게* 이 도구를 사용할지 결정합니다.
어떤 인자를 받는지, 어떤 상황에 호출해야 하는지를 명확하게 담아보세요.

In [ ]:
@tool
def search_employee(name: str) -> str:
    """TODO: 이 도구가 무엇을 하는지, 어떤 인자를 받는지 설명하세요."""
    return _search_employee_by_name(name)

In [ ]:
@tool
def search_product(query: str) -> str:
    """TODO: 이 도구가 무엇을 하는지, 어떤 인자를 받는지 설명하세요."""
    return _search_product_docs(query)

In [ ]:
@tool
def search_policy(query: str) -> str:
    """TODO: 이 도구가 무엇을 하는지, 어떤 인자를 받는지 설명하세요."""
    return _search_policy_docs(query)

## Step 5. 도구 직접 테스트 (LLM 없이)

에이전트를 만들기 전에 도구들이 제대로 동작하는지 먼저 확인합니다.

`@tool` 함수는 `.invoke()`로 직접 호출할 수 있습니다.
결과가 이상하다면 docstring보다 Step 3의 검색 기능을 먼저 점검하세요.

In [ ]:
print(search_employee.invoke({'name': '이서연'}))

In [ ]:
print(search_product.invoke({'query': 'TalkBridge Enterprise 요금제'}))

In [ ]:
print(search_policy.invoke({'query': '재택근무'}))

## Step 6. Sub-agent 만들기

각 sub-agent는 자신이 담당하는 도구와 역할을 가집니다.

**system_prompt를 직접 작성하세요.** 좋은 system_prompt에는 이런 내용이 들어갑니다:
- 이 에이전트가 무엇을 담당하는지 (역할)
- 어떤 도구를 어떻게 써야 하는지 (행동 지침)
- 어떻게 답변해야 하는지 (출력 형식)

In [ ]:
employee_agent = create_agent(
    llm,
    tools=[search_employee],
    system_prompt='TODO: 임직원 에이전트의 역할과 행동 지침을 작성하세요.'
)

In [ ]:
product_agent = create_agent(
    llm,
    tools=[search_product],
    system_prompt='TODO: 제품 에이전트의 역할과 행동 지침을 작성하세요.'
)

In [ ]:
policy_agent = create_agent(
    llm,
    tools=[search_policy],
    system_prompt='TODO: 규정 에이전트의 역할과 행동 지침을 작성하세요.'
)

## Step 7. Sub-agent를 Tool로 감싸기

Supervisor가 sub-agent를 호출하려면 sub-agent를 **Tool로 감싸야** 합니다.

docstring과 함수 서명은 제공되어 있습니다. **함수 본문을 직접 작성하세요.**

힌트: 각 sub-agent는 `.invoke({'messages': [...]})` 로 호출하고, 마지막 메시지의 `.content`를 반환합니다.

> Tool의 docstring이 중요한 이유: Supervisor LLM이 이 텍스트를 읽고 세 개의 Tool 중 어떤 것을 호출할지 결정합니다.

In [ ]:
@tool
def ask_employee_agent(query: str) -> str:
    """임직원 이름, 연락처, 부서, 담당업무, 프로젝트 현황 등 직원 관련 질문을 처리합니다."""
    # TODO: employee_agent를 호출하고 마지막 메시지의 content를 반환하세요
    pass


@tool
def ask_product_agent(query: str) -> str:
    """키키테크 제품 소개, 기능, 가격, 회사 개요 등 제품과 회사 관련 질문을 처리합니다."""
    # TODO: product_agent를 호출하고 마지막 메시지의 content를 반환하세요
    pass


@tool
def ask_policy_agent(query: str) -> str:
    """근무 규정, 복리후생, 행동강령, 보안 정책 등 사내 규정 관련 질문을 처리합니다."""
    # TODO: policy_agent를 호출하고 마지막 메시지의 content를 반환하세요
    pass

## Step 8. Supervisor 만들기

Supervisor는 사용자의 질문을 보고 세 sub-agent 중 알맞은 곳으로 라우팅합니다.
직접 검색하거나 답을 만들지 않고, **라우팅과 결과 전달**만 담당합니다.

**system_prompt를 직접 작성하세요.** Step 1의 Q3에서 적은 내용을 참고하면 좋습니다.

생각해볼 것:
- Supervisor는 어떤 질문을 어느 에이전트에게 보내야 할지 어떻게 구분할까요?
- 키키테크와 무관한 질문이 들어오면 어떻게 처리해야 할까요?

In [ ]:
supervisor = create_agent(
    llm,
    tools=[ask_employee_agent, ask_product_agent, ask_policy_agent],
    system_prompt='TODO: Supervisor의 역할과 라우팅 기준을 작성하세요.'
)

## Step 9. 실행해보기

세 가지 유형의 질문을 실행해 봅시다.
Supervisor가 각 질문을 올바른 sub-agent로 라우팅하는지 확인하세요.

만약 엉뚱한 에이전트가 호출되거나 답변이 이상하다면 → system_prompt나 docstring을 수정해 보세요.

In [ ]:
# 임직원 관련 질문 → 임직원 에이전트로 라우팅되어야 합니다
response = supervisor.invoke(
    {'messages': [{'role': 'user', 'content': '이서연 책임의 내선번호가 뭐야?'}]}
)
print(response['messages'][-1].content)

In [ ]:
# 제품 관련 질문 → 제품 에이전트로 라우팅되어야 합니다
response = supervisor.invoke(
    {'messages': [{'role': 'user', 'content': 'TalkBridge Enterprise 요금제가 어떻게 돼?'}]}
)
print(response['messages'][-1].content)

In [ ]:
# 규정 관련 질문 → 규정 에이전트로 라우팅되어야 합니다
response = supervisor.invoke(
    {'messages': [{'role': 'user', 'content': '재택근무는 일주일에 몇 번까지 가능해?'}]}
)
print(response['messages'][-1].content)

## 마무리: 설계와 구현 비교

Step 1에서 작성한 설계를 다시 꺼내보세요.

- 내가 설계한 에이전트 구성과 실제 구현이 같았나요?
- 다르다면 어디서 왜 바뀌었나요?
- 지금 구현을 개선한다면 어떤 부분을 바꾸고 싶나요?

---

**더 해보기**

1. **라우팅 실패 케이스 찾기**: 어떤 질문을 하면 Supervisor가 잘못된 에이전트로 보내나요? 왜 그럴까요?

2. **에이전트 추가**: 새로운 문서를 추가하고 새 sub-agent를 만들어 Supervisor에 연결해 보세요.

3. **경계 케이스**: 키키테크와 무관한 질문("오늘 날씨 어때?")을 넣으면 어떻게 동작하나요? system_prompt로 제어할 수 있나요?